In [ ]:
! pip install langchain-community

In [ ]:
! pip install tqdm

In [ ]:
! pip install chromadb

In [ ]:
! pip install python-dotenv

In [ ]:
! pip install transformers torch --upgrade

In [ ]:
! pip install langchain-groq

In [ ]:
! pip install  langchain-huggingface

In [ ]:
! pip install snowballstemmer

In [ ]:
! pip install sentence-transformers

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from chromadb import Schema, SparseVectorIndexConfig, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from chromadb import Schema, VectorIndexConfig
import chromadb.utils.embedding_functions as embedding_functions
from typing import List, Sequence
from langchain_community.embeddings import HuggingFaceEmbeddings
from chromadb.utils.embedding_functions import EmbeddingFunction
import uuid
from chromadb import Knn
from chromadb import Search

In [3]:
load_dotenv(find_dotenv(), override=False)

True

In [4]:
GROQ_API_KEY= os.getenv("GROQ_API_KEY")
HF_TOKEN= os.getenv("HF_TOKEN")
CHROMA_API_KEY = os.getenv("CHROMA_API_KEY")
LANG_SMITH_KEY = os.getenv("LANG_SMITH_KEY")

In [ ]:
schema = Schema()

In [5]:
client = chromadb.CloudClient(
  api_key=CHROMA_API_KEY,
  tenant="f6b39b05-f3b9-4614-896c-97cc6be82f15",
  database='hybrid_rag_alzheimers'
)

client.list_collections()

[Collection(name=medical_hybrid_rag_alzheimers),
 Collection(name=project_medical_hybrid_rag_alzheimers)]

In [ ]:
loader = DirectoryLoader('data/medical', glob="*.txt", show_progress=True, loader_cls=TextLoader)

In [ ]:
kb_docs = loader.load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=[". ", "? ", "! ", "\n\n", "\n", ", ", " "],
    keep_separator='end',
    is_separator_regex=False,
    chunk_size=500,
    chunk_overlap=75,
    length_function=len
)


In [ ]:
chunks = text_splitter.split_documents(kb_docs)

In [ ]:
print("Number of Documents:", len(kb_docs))
print()
print("Number of Chunks:", len(chunks))

In [7]:
schema = Schema()

In [8]:
schema = schema.create_index(
  key='sparse_vector_key',    
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)

In [9]:
model_name = "BAAI/bge-base-en-v1.5"

In [10]:
model_kwargs = {'device': 'cpu'}

In [11]:
encode_kwargs = {'normalize_embeddings': True}

In [ ]:
print(HF_TOKEN)

In [12]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.environ["HF_TOKEN"] = HF_TOKEN

In [14]:
hf_embedder  = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:

class HFLangChainEmbeddingFunction(EmbeddingFunction):
    def __init__(self, hf: HuggingFaceEmbeddings, use_query_head: bool = False):
        self.hf = hf
        self.use_query_head = use_query_head

    def __call__(self, input: Sequence[str]) -> List[List[float]]:
        # Chroma will pass a list of strings
        if self.use_query_head:
            return self.hf.embed_query(input)  # Some LC versions accept List[str]; if not, map()
        # Fallback to document head
        return self.hf.embed_documents(list(input))


In [16]:
embedding_fn = HFLangChainEmbeddingFunction(hf_embedder, use_query_head=False)

In [ ]:
schema = schema.create_index(
  key='sparse_vector_key',    
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)

In [ ]:
schema = schema.create_index(
    config=VectorIndexConfig(
        space="cosine",
        embedding_function=embedding_fn
    )
)

In [17]:
collection = client.get_or_create_collection(
  "project_medical_hybrid_rag_alzheimers",
  schema=schema, 
)

collection.count()

152

In [18]:
collection.peek()

{'ids': ['4dd41fd8-d8d7-4ace-a6ae-61dc3196d9c1',
  '2886bea3-154c-48a6-aa9b-c207a7ea84c8',
  '703644e2-47e3-4d10-b5e7-10bd17e9ba64',
  '067b4d71-a841-4652-835b-d50d5f05dad7',
  '97b2113f-7776-410d-8fdb-284b65852fcc',
  '58462c24-66b0-4bc3-a475-a8cad42e38ae',
  'e411dc5f-a29e-4317-a2d2-e06341d736bf',
  '120e96de-aff0-4a8a-b172-80f019cb0f50',
  '02239348-1e87-4772-8e59-18be463de28a',
  'bf1a0410-0856-4de2-8693-9e13b696451d'],
 'embeddings': array([[-0.00225256, -0.02085599, -0.04649356, ...,  0.08505344,
          0.00124186,  0.04453145],
        [ 0.0205027 , -0.03491891, -0.0299517 , ...,  0.05177085,
          0.07252148,  0.03284257],
        [-0.03316161, -0.03791902,  0.03644753, ...,  0.00901657,
          0.05613476, -0.01710049],
        ...,
        [-0.00930691,  0.00449889, -0.0280162 , ..., -0.0930873 ,
          0.01075573,  0.03139979],
        [ 0.04824402, -0.03885251, -0.04847791, ...,  0.10559314,
         -0.02290458,  0.08279263],
        [ 0.01575454,  0.01133357, 

In [19]:
print("Number of Documents:", len(kb_docs))
print()
print("Number of Chunks:", len(chunks))

NameError: name 'kb_docs' is not defined

In [ ]:
for chunk in chunks:
    print(chunk)

In [ ]:
collection.add(
    documents = [chunk.page_content for chunk in chunks],
    ids=[ str(uuid.uuid4()) for i in range(len(chunks)) ],
    metadatas= [chunk.metadata for chunk in chunks]
)

In [20]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json
import re
import ast
from typing import List, Tuple, Optional, Union
from typing import List, Dict, Any, Tuple, Iterable
from operator import itemgetter
from langchain_core.runnables import RunnableLambda
from chromadb import Search, Knn, Rrf, K  
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
print(GROQ_API_KEY)

In [21]:
QUERY_PROMPT_TEMPLATE = """ROLE: Multi-Query Expansion Generator for Hybrid Retrieval (Semantic + BM25) in MEDICAL (Alzheimer’s disease)

TASK
Rewrite the user’s question into EXACTLY 5 distinct queries optimized for both dense semantic search and sparse BM25. Output JSON with a "queries" array of 5 strings and an optional short "rationale". No extra text.

GUIDELINES
- BM25-friendly: Ensure some variants contain exact medical tokens, drug names, biomarker terms, test names, and likely keyword pairings; include AND-like co-occurrence phrasing.
- Semantic-friendly: Include natural, conversational paraphrases that capture intent, context, and outcomes.
- Safety: Do NOT invent facts, diagnoses, or treatment recommendations. Avoid speculative claims. Stay within the query’s scope.
- Distinctness: No trivial reorderings—each query should add different token mixes or intent angles.
- Quantity: ALWAYS output exactly 5 queries. If fewer are meaningful, include best variants that differ in phrasing and keyword composition.

STRICT OUTPUT FORMAT(DO NOT DEVIATE THE OUTPUT FORMAT)
[query1,query2,query3,query4,query5]

USER QUERY
{user_query}
"""

In [22]:
print(QUERY_PROMPT_TEMPLATE)

ROLE: Multi-Query Expansion Generator for Hybrid Retrieval (Semantic + BM25) in MEDICAL (Alzheimer’s disease)

TASK
Rewrite the user’s question into EXACTLY 5 distinct queries optimized for both dense semantic search and sparse BM25. Output JSON with a "queries" array of 5 strings and an optional short "rationale". No extra text.

GUIDELINES
- BM25-friendly: Ensure some variants contain exact medical tokens, drug names, biomarker terms, test names, and likely keyword pairings; include AND-like co-occurrence phrasing.
- Semantic-friendly: Include natural, conversational paraphrases that capture intent, context, and outcomes.
- Safety: Do NOT invent facts, diagnoses, or treatment recommendations. Avoid speculative claims. Stay within the query’s scope.
- Distinctness: No trivial reorderings—each query should add different token mixes or intent angles.
- Quantity: ALWAYS output exactly 5 queries. If fewer are meaningful, include best variants that differ in phrasing and keyword compositio

In [23]:
query_prompt_template = ChatPromptTemplate(
    messages=[
        QUERY_PROMPT_TEMPLATE
    ]
)

In [24]:
query_chat_model = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=1,
    max_tokens=None,           
    reasoning_format="hidden", 
    max_retries=2,
)

In [ ]:
# from langchain_core.output_parsers import JsonOutputParser

In [25]:
query_parser = StrOutputParser()

In [26]:
query_chain = query_prompt_template | query_chat_model | query_parser

In [27]:

def _strip_fences(text: str) -> str:
    # Remove ```...``` fences if present
    m = re.search(r"```(?:json|python)?\s*([\s\S]*?)```", text, flags=re.I)
    return m.group(1).strip() if m else text.strip()


In [28]:

def _loads_lenient(text: str) -> Union[dict, list]:
    """
    Try JSON first; if it fails (e.g., single quotes / Python dict),
    fall back to ast.literal_eval safely.
    """
    try:
        return json.loads(text)
    except Exception:
        # Attempt to extract the first JSON-ish object/list if surrounding prose exists
        # Try object
        m = re.search(r"(\{[\s\S]*\})", text)
        if m:
            chunk = m.group(1)
            # Try JSON again
            try:
                return json.loads(chunk)
            except Exception:
                # Try Python literal
                try:
                    return ast.literal_eval(chunk)
                except Exception:
                    pass
        # Try list
        m = re.search(r"(\[[\s\S]*\])", text)
        if m:
            chunk = m.group(1)
            try:
                return json.loads(chunk)
            except Exception:
                try:
                    return ast.literal_eval(chunk)
                except Exception:
                    pass
        # Final fallback: Python literal on whole text
        return ast.literal_eval(text)

def parse_llm_multiquery(raw_text: str) -> Tuple[List[str], Optional[str]]:
    """
    Accepts:
      - JSON array: ["q1","q2",...]
      - JSON object: {"queries":[...], "rationale":"..."}
      - Python-style dict/list with single quotes
      - With or without ```json fences
    Returns: (queries, rationale)
    """
    cleaned = _strip_fences(raw_text)
    data = _loads_lenient(cleaned)

    # Normalize shapes
    queries: List[str]
    rationale: Optional[str] = None

    if isinstance(data, list):
        queries = [str(x).strip() for x in data if isinstance(x, str)]
    elif isinstance(data, dict):
        # keys may not exist; be defensive
        q = data.get("queries")
        if isinstance(q, list):
            queries = [str(x).strip() for x in q if isinstance(x, str)]
        else:
            # If dict but no 'queries', try to coerce any list-like value
            list_like = next((v for v in data.values() if isinstance(v, list)), [])
            queries = [str(x).strip() for x in list_like if isinstance(x, str)]
        rationale_val = data.get("rationale")
        if isinstance(rationale_val, str):
            rationale = rationale_val.strip()
    else:
        raise ValueError("Unsupported LLM output format")

    if not queries:
        raise ValueError("No queries found in LLM output")

    # dedupe & normalize whitespace
    seen = set()
    normed = []
    for q in queries:
        key = " ".join(q.split()).lower()
        if key not in seen:
            seen.add(key)
            normed.append(" ".join(q.split()))
    return normed, rationale


In [29]:
DENSE_KEY = "#embedding"         
SPARSE_KEY = "sparse_vector_key"
TOP_K_PER_BRANCH = 4
FINAL_TOP_K = 10
DENSE_WEIGHT = 0.6
SPARSE_WEIGHT = 0.4

In [30]:

# Per-query retrieval (returns only the hybrid result)
def retrieve_one_hybrid(q):
    # Dense
    dense_rank = Knn(
        query=q,
        key="#embedding",
        return_rank=True,
    )
    dense_search = (
        Search()
        .rank(dense_rank)
        .select(K.DOCUMENT, K.SCORE, K.METADATA)
        .limit(5)
    )
    dense_results = collection.search(dense_search)

    # Sparse
    sparse_rank = Knn(
        query=q,
        key="sparse_vector_key",
        return_rank=True,
        limit=5
    )
    sparse_search = (
        Search()
        .rank(sparse_rank)
        .select(K.DOCUMENT, K.SCORE)
        .limit(5)
    )
    sparse_results = collection.search(sparse_search)

    # Hybrid (RRF)
    hybrid_rank = Rrf(
        ranks=[dense_rank, sparse_rank],
        weights=[0.7, 0.3],
        k=60
    )
    hybrid_search = (
        Search()
        .rank(hybrid_rank)
        .select(K.DOCUMENT, K.SCORE, K.METADATA)
        .limit(3)
        .limit(5)
    )
    hybrid_results = collection.search(hybrid_search)

    return hybrid_results  # keep it simple if you only need hybrid

In [31]:
_tok = AutoTokenizer.from_pretrained("BAAI/bge-reranker-base")
_mdl = AutoModelForSequenceClassification.from_pretrained("BAAI/bge-reranker-base")
_mdl.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=1, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Li

In [32]:

# --------------------- shape helpers ---------------------
def _to_doc_from_any(x: Any):
    """Extract text content from many possible shapes."""
    if x is None:
        return None
    if isinstance(x, dict):
        return x.get("document") or x.get("doc") or x.get("text") or x.get("value")
    if hasattr(x, "model_dump"):  # pydantic v2
        d = x.model_dump()
        return d.get("document") or d.get("doc") or d.get("text") or d.get("value")
    if hasattr(x, "dict"):  # pydantic v1
        d = x.dict()
        return d.get("document") or d.get("doc") or d.get("text") or d.get("value")
    if hasattr(x, "_asdict"):  # namedtuple
        d = x._asdict()
        return d.get("document") or d.get("doc") or d.get("text") or d.get("value")
    if isinstance(x, str):
        return x
    if isinstance(x, (list, tuple)) and x:
        # Heuristic: (document, score, metadata, id)
        return x[0] if isinstance(x[0], str) else None
    return None

def _to_id_from_any(x: Any, doc_text: str | None):
    """Prefer explicit id; fallback to metadata.id; else hash of doc."""
    if isinstance(x, dict):
        _id = x.get("id") or x.get("metadata", {}).get("id") or x.get("meta", {}).get("id")
        if _id is not None:
            return str(_id)
    if hasattr(x, "model_dump"):
        d = x.model_dump()
        _id = d.get("id") or d.get("metadata", {}).get("id") or d.get("meta", {}).get("id")
        if _id is not None:
            return str(_id)
    if hasattr(x, "dict"):
        d = x.dict()
        _id = d.get("id") or d.get("metadata", {}).get("id") or d.get("meta", {}).get("id")
        if _id is not None:
            return str(_id)
    if hasattr(x, "_asdict"):
        d = x._asdict()
        _id = d.get("id") or d.get("metadata", {}).get("id") or d.get("meta", {}).get("id")
        if _id is not None:
            return str(_id)
    if isinstance(x, (list, tuple)) and len(x) >= 4:
        return str(x[3])
    return str(hash(doc_text or ""))

def _to_meta_from_any(x: Any):
    if isinstance(x, dict):
        return x.get("metadata") or x.get("meta") or {}
    if hasattr(x, "model_dump"):
        d = x.model_dump()
        return d.get("metadata") or d.get("meta") or {}
    if hasattr(x, "dict"):
        d = x.dict()
        return d.get("metadata") or d.get("meta") or {}
    if hasattr(x, "_asdict"):
        d = x._asdict()
        return d.get("metadata") or d.get("meta") or {}
    if isinstance(x, (list, tuple)) and len(x) >= 3 and isinstance(x[2], dict):
        return x[2]
    return {}

def _to_score_from_any(x: Any, score_field: str = "score"):
    if isinstance(x, dict):
        if score_field in x: return x[score_field]
        for k in ("score", "similarity", "distance", "dist", "relevance"):
            if k in x: return x[k]
    if hasattr(x, "model_dump"):
        d = x.model_dump()
        if score_field in d: return d[score_field]
        for k in ("score", "similarity", "distance", "dist", "relevance"):
            if k in d: return d[k]
    if hasattr(x, "dict"):
        d = x.dict()
        if score_field in d: return d[score_field]
        for k in ("score", "similarity", "distance", "dist", "relevance"):
            if k in d: return d[k]
    if hasattr(x, "_asdict"):
        d = x._asdict()
        if score_field in d: return d[score_field]
        for k in ("score", "similarity", "distance", "dist", "relevance"):
            if k in d: return d[k]
    if isinstance(x, (list, tuple)) and len(x) >= 2 and isinstance(x[1], (int, float)):
        return x[1]
    return None

def _flatten_items_with_provenance(all_results: list) -> list[dict]:
    """
    Accepts per-subquery results (each can be:
      - list[dict|str|tuple]
      - {"results": [...]}
      - object with .results
      - Chroma-like dict with 'documents'/'ids'/'metadatas'/'distances'|'scores'
    Returns list of normalized dict hits with provenance fields.
    """
    pool = []
    for sub_idx, sub in enumerate(all_results):
        # Chroma query-like dict (ids, documents, metadatas, distances/scores)
        if isinstance(sub, dict) and "documents" in sub:
            docs = sub["documents"]
            ids = sub.get("ids")
            metas = sub.get("metadatas")
            dists = sub.get("distances") or sub.get("scores")
            # docs may be list-of-lists or flat list—handle both
            if docs and isinstance(docs[0], list):
                docs_list = docs[0]
                ids_list = ids[0] if ids and isinstance(ids[0], list) else ids
                metas_list = metas[0] if metas and isinstance(metas[0], list) else metas
                dists_list = dists[0] if dists and isinstance(dists[0], list) else dists
            else:
                docs_list, ids_list, metas_list, dists_list = docs, ids, metas, dists

            for rank_idx, _doc in enumerate(docs_list or []):
                _id = (ids_list[rank_idx] if ids_list else None)
                _meta = (metas_list[rank_idx] if metas_list else {})
                _score = (dists_list[rank_idx] if dists_list else None)
                pool.append({
                    "id": str(_id) if _id is not None else str(hash(_doc)),
                    "document": _doc,
                    "metadata": _meta or {},
                    "score": _score,
                    "_subquery_index": sub_idx,
                    "_rank_in_subquery": rank_idx,
                })
            continue

        # Generic list/object with .results
        items = (sub.get("results") if isinstance(sub, dict) and "results" in sub
                 else getattr(sub, "results", None) or sub)

        # Ensure iterable, avoid iterating strings/bytes
        if not isinstance(items, Iterable) or isinstance(items, (str, bytes)):
            items = [items]

        for rank_idx, raw in enumerate(items):
            doc = _to_doc_from_any(raw)
            if not doc:
                try:
                    doc = str(raw)
                except Exception:
                    continue
            pool.append({
                "id": _to_id_from_any(raw, doc),
                "document": doc,
                "metadata": _to_meta_from_any(raw),
                "score": _to_score_from_any(raw),
                "_subquery_index": sub_idx,
                "_rank_in_subquery": rank_idx,
            })
    return pool

def _minmax(xs):
    if not xs: return xs
    mn, mx = min(xs), max(xs)
    if mx == mn: return [0.5] * len(xs)
    return [(x - mn) / (mx - mn) for x in xs]

def _rrf_aggregate(ranks, k: int = 60) -> float:
    return sum(1.0 / (k + int(r)) for r in ranks)

def _invert_if_distance(score: float | None, is_distance: bool) -> float:
    if score is None: return 0.0
    return (1.0 / (1.0 + float(score))) if is_distance else float(score)

# --------------------- main function ---------------------
@torch.no_grad()
def rerank_multiquery_pool(
    original_query: str,
    all_results: list,       # 5 subquery results, each ~5 hits
    top_k: int = 5,
    mix_with_orig: bool = True,
    alpha: float = 0.25,
    score_field: str = "score",
    score_is_distance: bool = False,
    max_length: int = 512,
    device: str | None = None
) -> list[dict]:
    # 1) Flatten & normalize
    raw_pool = _flatten_items_with_provenance(all_results)
    if not raw_pool:
        return []

    # 2) Group by id (dedupe) & aggregate retrieval signal
    by_id = {}
    for it in raw_pool:
        _id = it["id"]
        sim = _invert_if_distance(it.get(score_field), score_is_distance)
        rnk = int(it.get("_rank_in_subquery", 0))
        row = by_id.setdefault(_id, {
            "id": _id,
            "document": it["document"],
            "metadata": it.get("metadata", {}),
            "provenance": [],
            "scores": [],
            "ranks": [],
        })
        row["provenance"].append({
            "subquery": it.get("_subquery_index"),
            "rank": rnk,
            "score": sim
        })
        row["scores"].append(sim)
        row["ranks"].append(rnk)

    pooled = []
    for _id, row in by_id.items():
        if not row["document"]:
            continue
        max_s = max(row["scores"]) if row["scores"] else 0.0
        rrf_s = _rrf_aggregate(row["ranks"], k=60) if row["ranks"] else 0.0
        agg = max_s + rrf_s
        pooled.append({
            "id": _id,
            "document": row["document"],
            "metadata": row["metadata"],
            "provenance": row["provenance"],
            "agg_score": agg
        })

    if not pooled:
        return []

    # 3) Cross-encoder rerank with original user query
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    _mdl.to(device)

    docs = [c["document"] for c in pooled]
    enc = _tok([original_query] * len(docs), docs,
               padding=True, truncation=True, max_length=max_length,
               return_tensors="pt").to(device)
    logits = _mdl(**enc).logits.view(-1)
    rr_scores = logits.detach().cpu().tolist()

    # 4) Optional blend with aggregated score
    if mix_with_orig:
        agg = [c["agg_score"] for c in pooled]
        final = [alpha * o + (1 - alpha) * r for o, r in zip(_minmax(agg), _minmax(rr_scores))]
    else:
        final = rr_scores

    for c, r_raw, r_fin in zip(pooled, rr_scores, final):
        c["raw_reranker"] = float(r_raw)
        c["rerank_score"] = float(r_fin)

    return sorted(pooled, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

In [51]:
MULTI_CONTEXT_PROMPT ="""
You are a careful, precise assistant. Use ONLY the provided context to answer. Your task is to answer the user Question from context, Sub Questions are provided to understand user Question better.

Answer the question strictly using the information in the provided context.
#########
{context}
#########
Question: {question}
#########
From the user query subqueries are derived by the system for you, Analyse the subquery and correlate with Question, but user question has the priority.
Sub Questions:{sub_query}
Instructions:
- Understand the context and the question semantically, not just by keywords.
- Give a direct and precise answer strictly based on the provided context.
- Do NOT justify your answer.
- Do NOT add information that is not present in the context.
- Do NOT mention the context explicitly.
- If the question cannot be answered from the context, or if there is no meaningful semantic similarity between the question and the context, respond exactly with:
  "This topic is not within my expertise. You may need to ask someone else."
########
"""

In [52]:
answer_chat_model = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=None,           
    reasoning_format="hidden", 
    max_retries=2,
    model_kwargs={      
        "top_p": 1.0,
      },
)

In [53]:
answer_prompt_template = ChatPromptTemplate(
    messages=[
        MULTI_CONTEXT_PROMPT
    ]
)

In [54]:
answer_parser = StrOutputParser()

In [55]:
answer_chain = answer_prompt_template | answer_chat_model | answer_parser

In [56]:
def _parse_multi(multi_query_output):
    queries, rationale = parse_llm_multiquery(multi_query_output)
    return {"queries": queries, "rationale": rationale}

In [57]:
make_multi = (
    {"user_query": RunnablePassthrough()}
    | RunnableParallel({
        "user_query": itemgetter("user_query"),
        "multi": itemgetter("user_query") | query_chain,
    })
    | RunnableLambda(lambda x: {
        "user_query": x["user_query"],
        **_parse_multi(x["multi"])
    })
)

In [58]:
parallel_stage = RunnableParallel({
    "user_query": itemgetter("user_query"),
    "queries": itemgetter("queries"),
    "rationale": itemgetter("rationale"),
    "all_results": itemgetter("queries") | RunnableLambda(retrieve_one_hybrid).map(),
}).with_config({"max_concurrency": 8})

In [59]:

rerank_stage = RunnableLambda(lambda x: {
    **x,
    "final_top5": rerank_multiquery_pool(
        original_query=x["user_query"],
        all_results=x["all_results"],  # list of lists (hybrid results per subquery)
        top_k=5,
        mix_with_orig=True,
        alpha=0.25,
        score_field="score",
        score_is_distance=False,
    )
})


In [60]:
def format_docs(docs):
    return "\n\n".join(
        (d.get("document") if isinstance(d, dict) else getattr(d, "page_content", str(d))).strip()
        for d in docs if d
    )

In [61]:
to_answer_inputs = RunnableLambda(lambda x: {
    "question": x["user_query"],       # original user query
    "sub_query": x["queries"],         # the 3–5 sub-queries
    "context": format_docs(x["final_top5"])  # reranked top-5 merged context
})

In [62]:
full_chain = make_multi | parallel_stage | rerank_stage | to_answer_inputs | answer_chain

In [46]:

query = input("Enter your question: ")
answer = await full_chain.ainvoke(query)


Enter your question:  what is dementia


In [47]:
print(answer)

This topic is not within my expertise. You may need to ask someone else.


In [49]:
print(to_answer_inputs)

RunnableLambda(lambda x: {'question': x['user_query'], 'sub_query': x['queries'], 'context': format_docs(x['final_top5'])})


In [63]:
# --- Pretty print for answer_chain input & output ---
from rich import print as rprint
from rich.panel import Panel
from rich.pretty import Pretty

# 1) Recompute ONLY the inputs to answer_chain (cheap)
query = input("Enter your question: ")
inputs_to_answer = await (make_multi | parallel_stage | rerank_stage | to_answer_inputs).ainvoke(query)

# 2) Pretty Print Inputs
rprint(Panel.fit(
    Pretty(inputs_to_answer, expand_all=True),
    title="🔹 INPUT to answer_chain",
    border_style="cyan"
))

# 3) Pretty Print Output (answer)
rprint(Panel.fit(
    Pretty(answer, expand_all=True) if isinstance(answer, (dict, list)) else answer,
    title="✅ OUTPUT from answer_chain",
    border_style="green"
))

Enter your question:  What is alzimers


╭─────────────────────────────────────────── 🔹 INPUT to answer_chain ────────────────────────────────────────────╮
│ {                                                                                                               │
│     'question': 'What is alzimers',                                                                             │
│     'sub_query': [                                                                                              │
│         'What is Alzheimer’s disease definition and symptoms',                                                  │
│         'Alzheimer’s disease causes and risk factors',                                                          │
│         'How is Alzheimer’s disease diagnosed (clinical criteria, biomarkers, imaging)',                        │
│         'What are the common treatments for Alzheimer’s disease (donepezil, rivastigmine, memantine)',          │
│         'What is the pathophysiology of Alzheimer’s disease (amyloid plaques, neurofibrillary tangles, tau      │
│ protein)'                                                                                                       │
│     ],                                                                                                          │
│     'context': 'Amnestic MCI has a greater than 90% likelihood of being associated with Alzheimer\'s.\nEarly    │
│ stage:\nIn people with Alzheimer\'s disease, the increasing impairment of learning and memory eventually leads  │
│ to a definitive diagnosis. In a small percentage, difficulties with language, executive functions, perception   │
│ (agnosia), or execution of movements (apraxia) are more prominent than memory problems. Alzheimer\'s disease    │
│ does not affect all memory capacities equally.\n\nNeurofibrillary tangles are aggregates of the                 │
│ microtubule-associated protein tau which has become hyperphosphorylated and accumulates inside neurons.         │
│ Although many older individuals develop some plaques and tangles as a consequence of aging, the brains of       │
│ people with Alzheimer\'s disease have a greater number of them in specific brain regions.\nThe two defining     │
│ proteopathies of Alzheimer\'s disease: AÎ² plaques (brown) and neurofibrillary (tau) tangles (black).\n\nThis   │
│ mixed pathology can complicate both diagnosis and the evaluation of clinical trials, which often target only    │
│ one of several potential contributors to dementia.\n\nBiochemistry\nMain article: Biochemistry of Alzheimer\'s  │
│ disease\nAmyloid beta (AÎ²)\nAlzheimer\'s disease has been identified as a protein misfolding disease, a        │
│ proteopathy, caused by the accumulation of abnormally folded AÎ² protein into amyloid plaques, and tau protein  │
│ into neurofibrillary tangles in the brain.\n\nThey define AD through three major stages: preclinical, mild      │
│ cognitive impairment (MCI), and Alzheimer\'s dementia. Diagnosis in the preclinical stage is complex and        │
│ focuses on asymptomatic individuals; the latter two stages describe individuals experiencing symptoms, along    │
│ with biomarkers, predominantly those for neuronal injury (mainly tau-related) and amyloid beta deposition. The  │
│ core clinical criteria itself rests on the presence of cognitive impairment without the presence of             │
│ comorbidities.\n\nDiagnosis\nPET scan of the brain of a person with Alzheimer\'s disease showing a loss of      │
│ function in the temporal lobe\nAlzheimer\'s disease (AD) can only be definitively diagnosed with autopsy        │
│ findings; in the absence of autopsy, clinical diagnoses of AD are "possible" or "probable", based on other      │
│ findings.'                                                                                                      │
│ }                                                                                                               │
╰────────────────────────────────────────────────────────

╭────────────────────── ✅ OUTPUT from answer_chain ───────────────────────╮
│ This topic is not within my expertise. You may need to ask someone else. │
╰──────────────────────────────────────────────────────────────────────────╯